# MAC Applied Artificial Intelligence · Week 4 Practical

**Google Colab · Student working notebook · 85–90 minutes per task**

This is a **starter notebook**, not a completed solution. Execute the cells **in order**. Keep your real computed outputs visible. Each task includes **student-dependent data** and intentionally unfinished decision cells.

**Rules:** Use your own student ID; don't change it after generating your dataset. Do not upload patient data. Submit **screenshots of outputs**, your **run token**, and a short interpretation in the separate Practical Submission app. Explanations without matching Colab evidence are incomplete. There are no downloadable external datasets.

**Academic integrity:** A generated notebook can still be solved using AI tools. The activity therefore requires live, individualized runs and interpretation, not just filled code.

## Task 1 — Numerical Data: Inspect, Repair, Scale and Validate
Concepts consolidated from original Labs 1–5: data leakage, inconsistent transformations, outliers, missing data and evaluation.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, accuracy_score
import hashlib

student_id = input('Enter your Student ID: ').strip()
assert len(student_id) >= 4 and student_id.replace('-','').replace('_','').isalnum(), 'Enter a valid student ID.'
seed = int(hashlib.sha256(student_id.encode()).hexdigest()[:8],16)
rng = np.random.default_rng(seed)
run_token = hashlib.sha256((student_id+'|MAC-W4|'+str(seed)).encode()).hexdigest()[:12].upper()
print('STUDENT ID:',student_id, '| RUN TOKEN:',run_token)
print('Take screenshot E0 showing your run token and student ID.')

### Activity N1 · Data-quality audit (Easy · 15 min)
This cell builds a **personalized tabular training sample** from the bundled `sklearn` diabetes dataset (no internet or external CSV required). Some problems are intentionally injected. **Do not assume extreme values are automatically errors.**

In [ ]:
from sklearn.datasets import load_diabetes
X0, y0 = load_diabetes(return_X_y=True, as_frame=True)
raw = X0.iloc[:220].copy().reset_index(drop=True)
raw['target'] = y0.iloc[:220].to_numpy()
row_a, row_b = rng.choice(len(raw), 2, replace=False)
raw.loc[row_a,'bmi'] = np.nan
raw.loc[row_b,'bp'] = np.nan
raw.loc[5,'bmi'] = raw['bmi'].max()*rng.uniform(7,11)
raw = pd.concat([raw,raw.iloc[[9]]],ignore_index=True)
print('Injected records:',row_a,row_b,' | duplicated original row: 9')
display(raw.head(12))
print('Missing counts:');display(raw.isna().sum().to_frame('missing'))
print('Duplicates:',raw.duplicated().sum())

**Student cell N1:** Write executable code to display the largest 5 BMI readings and calculate missing percentages for the `bmi` and `bp` columns. Record both outputs; explain which reading needs investigation. Screenshot **N1** must show the computed table.

In [ ]:
# TODO N1: run your own inspection; do not just type an expected value.
# Hint: raw['bmi'].nlargest(5); raw[['bmi','bp']].isna().mean()*100

# Enter your code below

### Activity N2 · Train-only imputation versus leakage (Medium · 20 min)
Use **fixed split indices** for a fair comparison. Experiment with median vs mean imputers and explain why fitting on full data is not recommended—even if the score differences are small.

In [ ]:
features=raw.drop(columns='target');target=raw['target']
train_idx,test_idx=train_test_split(np.arange(len(raw)),test_size=.25,random_state=seed%10000)
X_train,X_test=features.iloc[train_idx],features.iloc[test_idx]
y_train,y_test=target.iloc[train_idx],target.iloc[test_idx]
valid_cols=['bmi','bp','s5','age']
model=Ridge(alpha=1.0)
correct_imputer=SimpleImputer(strategy='median')
Xt=correct_imputer.fit_transform(X_train[valid_cols]);Xv=correct_imputer.transform(X_test[valid_cols])
model.fit(Xt,y_train)
correct_mae=mean_absolute_error(y_test,model.predict(Xv))
print('Train-only median imputation MAE:',round(correct_mae,3))
# Demonstration of INCORRECT workflow: full-data imputer fitting
wrong_imputer=SimpleImputer(strategy='median').fit(features[valid_cols])
wrong_train=wrong_imputer.transform(X_train[valid_cols]);wrong_test=wrong_imputer.transform(X_test[valid_cols])
wrong_model=Ridge(alpha=1.0).fit(wrong_train,y_train)
wrong_mae=mean_absolute_error(y_test,wrong_model.predict(wrong_test))
print('Full-data imputer MAE (leakage):',round(wrong_mae,3))

In [ ]:
# TODO N2: use a TRAIN-ONLY mean imputer; report its test MAE.
# YOUR output variable must be: mean_mae
# Fit on X_train[valid_cols]; transform train and test separately.
mean_mae = None
# Add implementation here

if mean_mae is not None:
    print('N2 train-only mean MAE:',round(float(mean_mae),3))

### Activity N3 · Scaling & outlier impact (Medium · 20 min)
Compare the same transaction values under three scalers. The output varies by student ID. Explain why ordinary values can be compressed when an extreme transaction exists.

In [ ]:
transactions=np.array([90,140,180,230,float(rng.integers(6000,14001))]).reshape(-1,1)
scale_table=pd.DataFrame({'Amount':transactions.ravel()})
for label,cls in [('MinMax',MinMaxScaler),('Standard',StandardScaler),('Robust',RobustScaler)]:
    scale_table[label]=cls().fit_transform(transactions).ravel()
display(scale_table.round(3))
scale_table.set_index('Amount')[['MinMax','Standard','Robust']].plot(marker='o',figsize=(7,3))
plt.title('Scaled amounts: derived from the displayed dataset');plt.grid(alpha=.2);plt.show()

In [ ]:
# TODO N3: change ONE transaction to half its value, then recompute all 3 scalers.
# Use a NEW variable transactions_modified; do not overwrite transactions.
# Display both before/after values for the first four customers.
transactions_modified = None
# Write your code here

### Activity N4 · Train/test consistency (Challenging · 20 min)
Fit StandardScaler on training only; compare with applying a *separately fitted* scaler to test data. Demonstrate the transformation mismatch (not every mismatch guarantees lower scores).

In [ ]:
scaler=StandardScaler().fit(X_train[valid_cols].fillna(X_train[valid_cols].median()))
train_filled=X_train[valid_cols].fillna(X_train[valid_cols].median())
test_filled=X_test[valid_cols].fillna(X_train[valid_cols].median())
correct_test=scaler.transform(test_filled)
incorrect_test=StandardScaler().fit_transform(test_filled)
print('Correct test feature means:',np.round(correct_test.mean(axis=0),3))
print('Incorrect separately-fitted test means:',np.round(incorrect_test.mean(axis=0),3))

In [ ]:
# TODO N4: Implement a sklearn Pipeline using SimpleImputer + StandardScaler + Ridge.
# Fit ONLY on train split and produce pipeline_mae for the test split.
pipeline_mae = None
# Write your code here
if pipeline_mae is not None:
    print('N4 pipeline test MAE:',round(float(pipeline_mae),3))

### Task 1 final run evidence
Run the following only **after** completing N1–N4. This signature helps the tutor match your outputs to your notebook; it is **not proof of independent work**.

In [ ]:
print('TASK 1 EVIDENCE | ID:',student_id,'| TOKEN:',run_token)
print('Train-only median MAE:',round(correct_mae,3))
print('Full-data median MAE:',round(wrong_mae,3))
print('Mean MAE:',round(float(mean_mae),3) if mean_mae is not None else 'INCOMPLETE')
print('Pipeline MAE:',round(float(pipeline_mae),3) if pipeline_mae is not None else 'INCOMPLETE')
print('Submission evidence: E0, N1, N2, N3, N4 (2-5 screenshots).')

**Written reflection:** Which single preprocessing decision changed your numeric evidence most? Cite the actual value in your screenshot. What would you check before applying this decision to unseen data?